# 01 - Data Understanding (YPerf JO28)

Objectif: comprendre la structure du dataset olympique, verifier sa qualite, et degager des insights initiaux pour orienter les predictions JO 2028.


## Plan de l'analyse

1. Charger et profiler les donnees
2. Verifier la qualite (valeurs manquantes, doublons, coherence temporelle)
3. Explorer les performances par pays, sport et genre
4. Identifier des signaux recents (momentum) vers Los Angeles 2028
5. Produire des premiers insights storytelling pour YPerf


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

sns.set_theme(style="darkgrid")


In [2]:
DATA_PATH = Path("../data/raw/olympics_dataset.csv")
assert DATA_PATH.exists(), f"Dataset introuvable: {DATA_PATH}"

df = pd.read_csv(DATA_PATH, low_memory=False)

# Quelques lignes brutes sont decalees d'une colonne: la valeur Year y est non numerique.
# On les ecarte pour fiabiliser les analyses (cf. reparation complete dans src/data_prep.py).
bad_year = pd.to_numeric(df["Year"], errors="coerce").isna()
print(f"Lignes malformees ecartees (Year non numerique): {int(bad_year.sum())}")
df = df[~bad_year].copy()
df["Year"] = df["Year"].astype(int)

print(f"Shape: {df.shape}")
print(f"Years: {int(df['Year'].min())} -> {int(df['Year'].max())}")
print(f"Countries (NOC): {df['NOC'].nunique()}")
print(f"Sports: {df['Sport'].nunique()}")

df.head(3)

Lignes malformees ecartees (Year non numerique): 1
Shape: (252564, 12)
Years: 1896 -> 2024
Countries (NOC): 234
Sports: 76


,player_id,Name,Sex,Team,NOC,Year,Season,City,Sport,Event,Medal,Unnamed: 11
0,0,A Dijiang,M,China,CHN,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,No medal,NaN
1,1,A Lamusi,M,China,CHN,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,No medal,NaN
2,2,Gunnar Aaby,M,Denmark,DEN,1920,Summer,Antwerpen,Football,Football Men's Football,No medal,NaN


## 1) Data dictionary rapide

In [3]:
data_dict = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
})

data_dict.sort_values("null_pct", ascending=False)


,column,dtype,null_pct,n_unique
11,Unnamed: 11,object,100.0,0
0,player_id,int64,0.0,235903
1,Name,object,0.0,129991
2,Sex,object,0.0,2
3,Team,object,0.0,1193
4,NOC,object,0.0,234
5,Year,int64,0.0,31
6,Season,object,0.0,1
7,City,object,0.0,23
8,Sport,object,0.0,76


## 2) Qualite des donnees

In [4]:
missing = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)
missing


,missing_pct
Unnamed: 11,100.0
player_id,0.0
Name,0.0
Sex,0.0
Team,0.0
NOC,0.0
Year,0.0
Season,0.0
City,0.0
Sport,0.0


In [5]:
dup_rows = df.duplicated().sum()
print(f"Duplicate rows: {dup_rows}")

key_cols = ["player_id", "Year", "Event"]
if all(c in df.columns for c in key_cols):
    dup_key = df.duplicated(subset=key_cols).sum()
    print(f"Duplicates on {key_cols}: {dup_key}")


Duplicate rows: 0
Duplicates on ['player_id', 'Year', 'Event']: 0


In [6]:
work = df.copy()
work["Medal"] = work["Medal"].fillna("None")
work["is_medal"] = (work["Medal"] != "None").astype(int)

medal_dist = (
    work["Medal"].value_counts(dropna=False)
    .rename_axis("Medal")
    .reset_index(name="count")
)
medal_dist


,Medal,count
0,No medal,213746
1,Bronze,13070
2,Gold,13002
3,Silver,12746


## 3) Exploration metier: pays, sports, genre

In [7]:
country_medals = (
    work.groupby("NOC", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

top_countries = country_medals.head(15)
fig = px.bar(top_countries, x="NOC", y="medals", title="Top 15 countries by historical medals")
fig.show()

top_countries


,NOC,medals
220,USA,16774
76,GBR,11998
71,FRA,11971
102,ITA,9351
81,GER,8866
13,AUS,8379
37,CAN,7907
106,JPN,7721
92,HUN,6621
197,SWE,6422


In [8]:
sport_medals = (
    work.groupby("Sport", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

fig = px.bar(sport_medals.head(15), x="Sport", y="medals", title="Top 15 sports by medals volume")
fig.update_layout(xaxis_tickangle=-35)
fig.show()

sport_medals.head(15)


,Sport,medals
8,Athletics,43294
38,Gymnastics,26707
63,Swimming,26416
58,Shooting,12580
54,Rowing,11625
34,Fencing,11558
22,Cycling,10859
36,Football,7906
75,Wrestling,7734
57,Sailing,7266


In [9]:
sex_year = (
    work.groupby(["Year", "Sex"], as_index=False)
    .size()
    .rename(columns={"size": "entries"})
)

total_year = sex_year.groupby("Year", as_index=False)["entries"].sum().rename(columns={"entries": "total"})
sex_year = sex_year.merge(total_year, on="Year", how="left")
sex_year["share"] = sex_year["entries"] / sex_year["total"]

fig = px.line(sex_year, x="Year", y="share", color="Sex", markers=True, title="Participation share by gender over time")
fig.show()

sex_year.tail(10)


,Year,Sex,entries,total,share
51,2008,F,5816,13602,0.427584
52,2008,M,7786,13602,0.572416
53,2012,F,5815,12920,0.450077
54,2012,M,7105,12920,0.549923
55,2016,F,6223,13688,0.454632
56,2016,M,7465,13688,0.545368
57,2020,F,7266,15120,0.480556
58,2020,M,7854,15120,0.519444
59,2024,F,7312,14892,0.491002
60,2024,M,7580,14892,0.508998


## 4) Signals recents pour JO 2028

On compare la periode recente (2016-2024) avec l'historique precedent pour detecter des progressions.


In [10]:
recent_start = 2016

country_year = (
    work.groupby(["NOC", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

hist = country_year[country_year["Year"] < recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg_medals"})
recent = country_year[country_year["Year"] >= recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg_medals"})

momentum = hist.merge(recent, on="NOC", how="outer").fillna(0)
momentum["delta"] = momentum["recent_avg_medals"] - momentum["hist_avg_medals"]
momentum["delta_pct"] = np.where(
    momentum["hist_avg_medals"] > 0,
    momentum["delta"] / momentum["hist_avg_medals"],
    np.nan,
)

momentum_filtered = momentum[momentum["recent_avg_medals"] >= 5].sort_values("delta", ascending=False)
momentum_filtered.head(20)


,NOC,hist_avg_medals,recent_avg_medals,delta,delta_pct
171,ROC,0.000000,561.000000,561.000000,NaN
13,AUS,252.846154,601.666667,348.820513,1.379576
106,JPN,280.952381,607.000000,326.047619,1.160508
30,BRA,145.500000,466.666667,321.166667,2.207331
220,USA,531.296296,809.666667,278.370370,0.523946
71,FRA,361.464286,616.666667,255.202381,0.706024
37,CAN,248.269231,484.000000,235.730769,0.949497
42,CHN,318.666667,553.333333,234.666667,0.736402
65,ESP,206.545455,439.333333,232.787879,1.127054
102,ITA,279.214286,511.000000,231.785714,0.830136


In [11]:
sport_year = (
    work.groupby(["Sport", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

sport_hist = sport_year[sport_year["Year"] < recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg"})
sport_recent = sport_year[sport_year["Year"] >= recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg"})

sport_momentum = sport_hist.merge(sport_recent, on="Sport", how="outer").fillna(0)
sport_momentum["delta"] = sport_momentum["recent_avg"] - sport_momentum["hist_avg"]

sport_momentum.sort_values("delta", ascending=False).head(15)


,Sport,hist_avg,recent_avg,delta
6,Artistic Gymnastics,0.000000,1148.500000,1148.500000
8,Athletics,1289.857143,2392.666667,1102.809524
63,Swimming,772.392857,1596.333333,823.940476
30,Cycling Track,0.000000,439.000000,439.000000
32,Equestrian,0.000000,417.500000,417.500000
18,Canoe Sprint,0.000000,378.000000,378.000000
56,Rugby Sevens,0.000000,309.333333,309.333333
36,Football,241.230769,544.666667,303.435897
22,Cycling,364.000000,667.000000,303.000000
26,Cycling Road,0.000000,246.500000,246.500000


## 5) Storytelling: premiers insights YPerf

Adapter ces messages apres execution des cellules ci-dessus:

- Insight 1 (macro): quels pays dominent structurellement le classement medailles.
- Insight 2 (dynamique): quels pays montrent le plus de momentum depuis 2016.
- Insight 3 (portfolio): quels sports concentrent le volume medailles et lesquels accelerent.
- Insight 4 (equite/performance): evolution de la participation F/M et impact potentiel sur les opportunites medailles.

### Pays/sports a suivre pour 2028 (template)
- Pays etablis a surveiller: top volume + stabilite multi-editions
- Pays en progression: top `delta` recent vs historique
- Sports strategiques: top `delta` sport + adequation historique des pays cibles

### Implications ML
- Cible recommandee: nb de medailles par (NOC, Sport, Year)
- Features prioritaires: lags medailles, rolling means, volume d'entrees, ratio F, effets periode recente
- Validation: split temporel strict (train <= 2016, valid = 2020 ou 2024 selon scenario)


In [12]:
out_dir = Path("../reports/metrics")
out_dir.mkdir(parents=True, exist_ok=True)

top_countries.to_csv(out_dir / "eda_top_countries.csv", index=False)
sport_medals.head(20).to_csv(out_dir / "eda_top_sports.csv", index=False)
momentum_filtered.head(20).to_csv(out_dir / "eda_country_momentum.csv", index=False)

print(f"EDA tables exported to: {out_dir.resolve()}")


EDA tables exported to: /Users/alx/Ynov/B3/Projet-Fil-Rouge-JO28/reports/metrics
